## Import Necessary Libraries and Set Paths

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
HDFS_URI = "hdfs://localhost:9000"

# os.environ["JAVA_HOME"] = os.path.abspath("java-se-8u44-ri")
# os.environ["SPARK_HOME"] = os.path.abspath("spark-3.5.5-bin-hadoop3")
# os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

INP_FILE_PATH = os.path.abspath("./asr.csv")
OUT_FILE_PATH = os.path.abspath(".")

## Create a SparkSession in Python

In [3]:
spark = SparkSession.builder\
    .appName("CountPreviousWeekSale") \
    .config("spark.hadoop.fs.defaultFS", HDFS_URI) \
    .getOrCreate()

print(spark.version)

25/04/11 19:04:55 WARN Utils: Your hostname, Luminous resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp0s20f3)
25/04/11 19:04:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/11 19:04:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.5


# 2.2 Tính số lượng sản phẩm đã bán trong 7 ngày qua của mỗi SKU, báo cáo mỗi thứ Hai hàng tuần

**Sử dụng Apache Spark**:
1. Thực hiện tính toán để tính tổng số lượng (**Qty**) của các mặt hàng được vận chuyển
từng **SKU** trong 7 ngày qua nhưng chỉ báo cáo tổng số này vào thứ Hai hàng tuần.
2. Mục tiêu là cung cấp ảnh chụp nhanh hàng tuần về tổng số lượng được vận chuyển cho mỗi **SKU**
trong tuần qua.
3. Sắp xếp kết quả theo thứ tự tăng dần theo **report_date**, tiếp theo là **SKU**.

Đọc dữ liệu tập asr.csv

In [4]:
dataset = spark.read.csv(f"file:///{INP_FILE_PATH}", sep=',', header=True, inferSchema=True)

In [5]:
dataset.show()

+-----+-------------------+--------+--------------------+----------+--------------+------------------+--------+-------------------+-------------+----+----------+--------------+---+--------+------+-----------+--------------+----------------+------------+--------------------+-----+------------+-----------+
|index|           Order ID|    Date|              Status|Fulfilment|Sales Channel |ship-service-level|   Style|                SKU|     Category|Size|      ASIN|Courier Status|Qty|currency|Amount|  ship-city|    ship-state|ship-postal-code|ship-country|       promotion-ids|  B2B|fulfilled-by|Unnamed: 22|
+-----+-------------------+--------+--------------------+----------+--------------+------------------+--------+-------------------+-------------+----+----------+--------------+---+--------+------+-----------+--------------+----------------+------------+--------------------+-----+------------+-----------+
|    0|405-8078784-5731545|04-30-22|           Cancelled|  Merchant|     Amazon.in

Lọc bộ dữ liệu chỉ bao gồm các mặt hàng đã được ship và có quantity khác 0

In [6]:
dataset = dataset.filter((f.col("Status") == "Shipped") & (f.col("Qty") != 0))

## Thống kê dữ liệu Date trong bộ dữ liệu

Cho biết khoảng dữ liệu có trong cột Date.

In [7]:
df = dataset.withColumn("Date", f.to_date(f.col("Date"), "MM-dd-yy"))
df = df.withColumn("Month", f.month(f.col("Date")))

In [8]:
max_min_dates = df.groupBy("Month").agg(
    f.min("Date").alias("Min_Date"),
    f.max("Date").alias("Max_Date"),
    f.countDistinct(f.day("Date")).alias("Num_Days")
)

In [9]:
max_min_dates.orderBy("Month").show()

+-----+----------+----------+--------+
|Month|  Min_Date|  Max_Date|Num_Days|
+-----+----------+----------+--------+
|    3|2022-03-31|2022-03-31|       1|
|    4|2022-04-01|2022-04-30|      30|
|    5|2022-05-01|2022-05-31|      31|
|    6|2022-06-01|2022-06-29|      29|
+-----+----------+----------+--------+



### Cho biết ngày báo cáo cho mỗi đối tượng dữ liệu

In [10]:
sorted_days = df.select(f.col("Date"))\
                .distinct()\
                .orderBy(f.desc(f.col("Date")))
sorted_days = sorted_days.withColumn("day_of_week",f.dayofweek(f.col("Date")))

Hiển thị ngày trong tuần.

In [11]:
sorted_days.show()

+----------+-----------+
|      Date|day_of_week|
+----------+-----------+
|2022-06-29|          4|
|2022-06-28|          3|
|2022-06-27|          2|
|2022-06-26|          1|
|2022-06-25|          7|
|2022-06-24|          6|
|2022-06-23|          5|
|2022-06-22|          4|
|2022-06-21|          3|
|2022-06-20|          2|
|2022-06-19|          1|
|2022-06-18|          7|
|2022-06-17|          6|
|2022-06-16|          5|
|2022-06-15|          4|
|2022-06-14|          3|
|2022-06-13|          2|
|2022-06-12|          1|
|2022-06-11|          7|
|2022-06-10|          6|
+----------+-----------+
only showing top 20 rows



In [12]:
sorted_days= sorted_days.withColumn("days_to_next_monday",
                   f.expr("CASE WHEN day_of_week = 2 THEN 7 ELSE (9 - day_of_week) % 7 END"))

# Tính report_date bằng cách cộng số ngày vào Date
sorted_days = sorted_days.withColumn("report_date", f.date_add(f.col("Date"), f.col("days_to_next_monday")))

# Hiển thị kết quả
report_date = sorted_days.select("Date", "report_date")

In [13]:
report_date.show()

+----------+-----------+
|      Date|report_date|
+----------+-----------+
|2022-06-29| 2022-07-04|
|2022-06-28| 2022-07-04|
|2022-06-27| 2022-07-04|
|2022-06-26| 2022-06-27|
|2022-06-25| 2022-06-27|
|2022-06-24| 2022-06-27|
|2022-06-23| 2022-06-27|
|2022-06-22| 2022-06-27|
|2022-06-21| 2022-06-27|
|2022-06-20| 2022-06-27|
|2022-06-19| 2022-06-20|
|2022-06-18| 2022-06-20|
|2022-06-17| 2022-06-20|
|2022-06-16| 2022-06-20|
|2022-06-15| 2022-06-20|
|2022-06-14| 2022-06-20|
|2022-06-13| 2022-06-20|
|2022-06-12| 2022-06-13|
|2022-06-11| 2022-06-13|
|2022-06-10| 2022-06-13|
+----------+-----------+
only showing top 20 rows



In [14]:
df_joined = df.join(
    report_date.select("Date", "report_date"),
    on="Date",
    how="left"
)

Lọc ra những dữ liệu có ngày báo cáo nằm trong khoảng từ ngày sớm nhất và ngày xa nhất trong bộ dữ liệu. Do những dữ liệu có ngày báo cáo chưa tới hoặc chưa có đủ dữ liệu để thực hiện báo cáo.
- Ví dụ:
  - Ngày sớm nhất được ghi nhận **2022-06-29** có ngày báo cáo là ngày **2022-07-04**, ko có dữ liệu ngày 30, 1, 2, 3 để ghi nhận.
  - Ngày xa nhất được ghi nhận **2022-03-31** có ngày báo cáo là ngày **2022-04-04**, ko có dữ liệu ngày 28, 29, 30 để ghi nhận.

In [15]:
date1 = f.to_date(f.lit("2022-06-27"))
date2 = f.to_date(f.lit("2022-04-11"))

In [16]:
filtered_df = df_joined.filter((f.col("report_date") >= date2) & (f.col("report_date") <= date1))

Lọc ra các mẫu SKU. Việc lọc này có thể không cần thiết do mã SKU mặc định bao gồm đa dạng thông tin ngoài mã sản phẩm như khối lượng, năm, màu sắc, ... Để đưa ra kết quả thống kê khái quát, nhóm chọn việc bỏ qua những thông số này và chỉ xem những sản phẩm cùng loại là 1.

In [91]:
#optional
# filtered_df = filtered_df.withColumn("SKU", f.split(f.col("SKU"), "-")[0])

In [17]:
# Gom nhóm theo report_date và sku, tính tổng quantity
result = filtered_df.groupBy("report_date", "SKU")\
                      .agg({"Qty": "sum"})\
                      .withColumnRenamed("sum(Qty)", "total_quantity")

In [18]:
# Sắp xếp theo report_date (tăng dần) và sku (tăng dần)
result = result.orderBy("report_date", "SKU")

In [19]:
# Hiển thị kết quả
result.show()

+-----------+----------------+--------------+
|report_date|             SKU|total_quantity|
+-----------+----------------+--------------+
| 2022-04-11|     AN201-RED-M|             1|
| 2022-04-11|  AN202-ORANGE-S|             1|
| 2022-04-11|  AN205-YELLOW-S|             2|
| 2022-04-11|   AN206-GREEN-M|             1|
| 2022-04-11| AN208-MUSTARD-M|             1|
| 2022-04-11|AN208-MUSTARD-XL|             1|
| 2022-04-11|  AN209-BIEGE-XL|             1|
| 2022-04-11|   AN213-BROWN-S|             2|
| 2022-04-11|   BL008-61RED-B|             1|
| 2022-04-11|    BL015-63PINK|             1|
| 2022-04-11|   BL050-83RED-A|             1|
| 2022-04-11|     BL079-87RED|             1|
| 2022-04-11|       BL086-XXL|             1|
| 2022-04-11|         BL096-S|             1|
| 2022-04-11|         BL102-L|             1|
| 2022-04-11|         BL102-M|             1|
| 2022-04-11|        BL103-XS|             1|
| 2022-04-11|       BL103-XXL|             1|
| 2022-04-11|        BL104-XS|    

### Format dữ liệu theo yêu cầu

In [20]:
result = result.withColumn("report_date", f.date_format(f.col("report_date"), "dd-MM-yyyy"))
result = result.withColumnRenamed("SKU", "sku")

In [21]:
# Ghi kết quả vào file CSV
result.toPandas().to_csv("output.csv", index=False)

In [22]:
spark.sparkContext.stop()
spark.stop()